# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("data/E Commerce Dataset.xlsx"),
    Path("../E Commerce Dataset.xlsx"),
    Path("E Commerce Dataset.xlsx")]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

读取文件：data\E Commerce Dataset.xlsx
数据形状：5630 行，20 列


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [2]:
df.info()

# 答案：
# 1.一行代表一个客户（用户）的完整画像信息，涵盖了该客户的基本属性、行为特征、偏好设置以及消费记录等20个维度的信息。例如 CustomerID=50001 的行记录了一位女性客户的全部信息：她使用 Mobile Phone 登录、居住在3线城市、距离仓库6公里、偏好借记卡支付、在App上花费3小时、注册了3台设备、偏好购买笔记本电脑及配件、满意度评分为2、单身、有9个地址、上月有投诉、订单金额同比增长11%、使用了1张优惠券、上月下单1次、距上次订单5天、获得返现159.93
# 2.CustomerID 是用户唯一标识。从示例数据可以看到 50001、50002、50003 等连续递增的编号，每个客户对应一个独立的 ID。
# 3.Churn 是流失分析的目标变量。该字段为二分类变量（示例中值为 0 或 1），其中 1 表示客户已流失，0 表示客户未流失。其他 19 个字段均可作为特征变量，用于构建流失预测模型。

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [3]:
# TODO：创建 missing_report
# 提示：df.isna().sum() 统计缺失数量；df.isna().mean() 计算缺失比例。


missing_report = pd.DataFrame({
    '缺失数量': df.isnull().sum().values,
    '缺失比例': (df.isnull().sum().values / len(df) * 100).round(2)
}, index=df.columns)
missing_report['缺失比例'] = missing_report['缺失比例'].astype(str) + '%'
missing_report = missing_report.sort_values(by='缺失数量', ascending=False)
print(missing_report.to_string())
# display(missing_report)

                             缺失数量   缺失比例
DaySinceLastOrder             307  5.45%
OrderAmountHikeFromlastYear   265  4.71%
Tenure                        264  4.69%
OrderCount                    258  4.58%
CouponUsed                    256  4.55%
HourSpendOnApp                255  4.53%
WarehouseToHome               251  4.46%
CustomerID                      0   0.0%
MaritalStatus                   0   0.0%
Complain                        0   0.0%
NumberOfAddress                 0   0.0%
PreferedOrderCat                0   0.0%
SatisfactionScore               0   0.0%
Churn                           0   0.0%
NumberOfDeviceRegistered        0   0.0%
Gender                          0   0.0%
PreferredPaymentMode            0   0.0%
CityTier                        0   0.0%
PreferredLoginDevice            0   0.0%
CashbackAmount                  0   0.0%


### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [4]:
print("重复值检查")
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df['CustomerID'].duplicated().sum()
print("完全重复行数：", duplicate_rows)
print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为可能是多期快照、系统合并或录入错误导致，需先抽样确认重复记录的字段差异和业务背景，再决定去重策略（如保留最新记录）

重复值检查
完全重复行数： 0
CustomerID 重复数量： 0


---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [5]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]
print("\n" + "=" * 60)
print("任务2: 用中位数填补数值型缺失值")
print("=" * 60)

# 识别数值型列(排除 CustomerID 标识列)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_to_fill = [col for col in numeric_cols if col != 'CustomerID']

print(f"需填补的数值型字段: {numeric_cols_to_fill}")

for col in numeric_cols_to_fill:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        median_val = df[col].median()
        print(f"  {col}: 缺失{missing_count}个, 中位数={median_val}")
        df[col] = df[col].fillna(median_val)

print(f"填补后数值型字段缺失总数: {df[numeric_cols_to_fill].isnull().sum().sum()}")
# TODO：循环填充每列的中位数
# for col in numeric_missing_cols:
#     ...

# TODO：输出上述字段剩余的缺失值数量
# print(df[numeric_missing_cols].isna().sum())


任务2: 用中位数填补数值型缺失值
需填补的数值型字段: ['Churn', 'Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
  Tenure: 缺失264个, 中位数=9.0
  WarehouseToHome: 缺失251个, 中位数=14.0
  HourSpendOnApp: 缺失255个, 中位数=3.0
  OrderAmountHikeFromlastYear: 缺失265个, 中位数=15.0
  CouponUsed: 缺失256个, 中位数=1.0
  OrderCount: 缺失258个, 中位数=2.0
  DaySinceLastOrder: 缺失307个, 中位数=3.0
填补后数值型字段缺失总数: 0


### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [6]:
# 场景：缺失非随机（MNAR）且缺失比例较高时。例如流失客户（Churn=1）的
# Tenure 缺失率显著高于留存客户（Churn=0），说明"缺失本身携带信息"——
# 可能流失客户注册时间太短系统未记录，或长期未活跃导致数据丢失。
# 此时若用全局中位数统一填充，会抹平两组的真实差异，使流失客户的 Tenure
# 被人为拉高，导致模型误判。
# 更合适的处理思路：
#   ① 先按目标变量（如 Churn）分组，分别计算组内中位数进行填充；
#   ② 同时添加"Tenure_是否缺失"二元指示列，保留缺失本身的信号；
#   ③ 若缺失比例超过50%，考虑将该字段降级为二元特征或从模型中剔除，
#     避免单一数值主导样本分布。

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [7]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64


### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [8]:
# TODO：完成类别标准化
# df["PreferredLoginDevice"] = ...
# df["PreferredPaymentMode"] = ...
# df["PreferedOrderCat"] = ...

# TODO：重新检查类别频数
# for col in category_cols:
#     print(f"\n{col}")
#     print(df[col].value_counts())

print("\n" + "=" * 60)
print("类别标准化")
print("=" * 60)

# 定义标准化映射规则
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Phone": "Mobile Phone"
})
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})
df["PreferedOrderCat"] = df["PreferedOrderCat"].replace({
    "Mobile": "Mobile Phone"
})

# 重新检查类别频数
category_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())



类别标准化

PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [9]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [10]:
# TODO：调用函数检查三个字段
display(iqr_outlier_summary(df["WarehouseToHome"]))
display(iqr_outlier_summary(df["OrderCount"]))
display(iqr_outlier_summary(df["CashbackAmount"]))

# 结论：候选异常值不能直接删除，因为 IQR 方法仅基于统计分布划定边界，
# 识别出的"异常"可能对应真实的业务场景（如大客户的高订单量、偏远地区的长配送距离、
# 高价值客户的较高返现金额）。直接删除会损失有效信息并引入样本偏差。
# 正确做法是先标记异常值，结合业务逻辑判断其合理性，再决定保留、修正或删除。

Q1         9.00
Q3        20.00
下限        -7.50
上限        36.50
候选异常值数量    2.00
dtype: float64

Q1          1.00
Q3          3.00
下限         -2.00
上限          6.00
候选异常值数量   703.00
dtype: float64

Q1        145.77
Q3        196.39
下限         69.84
上限        272.33
候选异常值数量   438.00
dtype: float64

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [11]:
# TODO：完成业务规则检查
# rules = {
#     "使用时长小于 0": ...,
#     "仓库距离小于 0": ...,
#     "订单数小于或等于 0": ...,
#     "返现金额小于 0": ...,
# }
# pd.Series(rules)
print("\n" + "=" * 60)
print("业务规则检查")
print("=" * 60)
rules = {
    "使用时长小于 0": (df["Tenure"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}
print(pd.Series(rules))


业务规则检查
使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
dtype: int64


---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [12]:
# TODO：完成验收
assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

print("数据清洗验收通过。")

数据清洗验收通过。


### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [13]:
OUTPUT_PATH = Path("output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"已导出：{OUTPUT_PATH.resolve()}")

已导出：C:\Users\wangy\notebook\output\ecommerce_customer_cleaned.csv


---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？